# VascuMamba-Spot — End-to-End Google Colab GPU Pipeline
Run full feature extraction, Phase I rPPG/BIOPAC audit, 4M-frame SSL pretraining, LOSO benchmarks, ablations (`ABL-1`..`ABL-10`), and automated LaTeX table/figure generation on a Colab GPU (T4 / L4 / A100).

In [ ]:
# 1. Check GPU & Mount Google Drive
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone Repository & Install Dependencies (Uses built-in Triton GPU scan by default)
REPO_URL = 'https://github.com/Vishal442005/VascuMamba-Spot.git'  # Update if private/different
!git clone {REPO_URL} /content/VascuMamba-Spot || (cd /content/VascuMamba-Spot && git pull)
%cd /content/VascuMamba-Spot
!pip install -q "numpy<2" -e ".[dev]"
# Optional on Python <=3.12: !pip install -q --no-build-isolation causal-conv1d>=1.2 mamba-ssm>=2.0

In [ ]:
# 3. Verify Unit Tests & Triton/CUDA Mamba Availability
!OMP_NUM_THREADS=1 PYTEST_DISABLE_PLUGIN_AUTOLOAD=1 pytest -v
!python -c "import torch; from vascumamba_spot.models.ssm_scan import is_cuda_mamba_available, resolve_ssm_backend; from vascumamba_spot.models.triton_scan import is_triton_mamba_available; print('CUDA:', torch.cuda.is_available(), 'triton_mamba:', is_triton_mamba_available(), 'cuda_mamba:', is_cuda_mamba_available(), 'active_backend:', resolve_ssm_backend('auto'))"

In [ ]:
# 4. Configure Licensed Dataset Paths on Google Drive
DATASET_NAME = 'casme3'  # casme3 | casme_sq | casme2 | samm | samm_lv | smic_hs | smic_vis | smic_nir | mmew
DATA_ROOT = '/content/drive/MyDrive/ME_Datasets/CASME3'
OUT_DIR = '/content/drive/MyDrive/VascuMamba_Outputs'

!mkdir -p data cache checkpoints {OUT_DIR}
!python scripts/prepare_manifests.py --dataset {DATASET_NAME} --data-root {DATA_ROOT} --out-manifest data/{DATASET_NAME}.jsonl
!python scripts/extract_features.py --manifest data/{DATASET_NAME}.jsonl --out-cache cache/{DATASET_NAME}.h5 --diagnostics-json {OUT_DIR}/{DATASET_NAME}_rppg_diag.json
!python scripts/audit_rppg_snr.py --manifest data/{DATASET_NAME}.jsonl --cache cache/{DATASET_NAME}.h5 --out-report {OUT_DIR}/{DATASET_NAME}_phase1_audit.json

In [ ]:
# 5. Optional 4M-Frame Neuro-Vascular SSL Pretraining (CAS(ME)^3 Unlabeled)
# !python scripts/prepare_manifests.py --dataset casme3_unlabeled --data-root {DATA_ROOT} --out-manifest data/casme3_unlabeled.jsonl
# !python scripts/extract_features.py --manifest data/casme3_unlabeled.jsonl --out-cache cache/casme3_unlabeled.h5
# !python scripts/pretrain_ssl.py --manifest data/casme3_unlabeled.jsonl --cache cache/casme3_unlabeled.h5 --out-ckpt checkpoints/ssl_backbone.pt

In [ ]:
# 6. Run LOSO Benchmarks, Ablations (ABL-1..ABL-10), Scaling, & Generate Paper Assets
!python scripts/train_loso.py --manifest data/{DATASET_NAME}.jsonl --cache cache/{DATASET_NAME}.h5 --seeds 42 101 2026 --out-json {OUT_DIR}/{DATASET_NAME}_loso.json
!python scripts/run_ablations.py --manifest data/{DATASET_NAME}.jsonl --cache cache/{DATASET_NAME}.h5 --out-json {OUT_DIR}/{DATASET_NAME}_ablations.json
!python scripts/benchmark_scaling.py --lengths 500 1000 2500 5000 10000 15000 --out-json {OUT_DIR}/gpu_scaling.json --out-fig paper/figures/fig4_scaling.pdf
!python scripts/generate_paper_assets.py --ablation-json {OUT_DIR}/{DATASET_NAME}_ablations.json --spotting-jsons {OUT_DIR}/{DATASET_NAME}_loso.json --out-dir paper